# CryptEdu Local AI Tutor — Training Notebook
**Qwen2.5-7B-Instruct fine-tuning on SPM Mathematics**

| Cell | Purpose |
|------|---------|
| 1 | Setup: install deps, mount Drive, GPU check |
| 2 | Document ingestion: OCR with tesseract, load textbooks |
| 3 | Exam pairing: pair_exams Q&A matching |
| 4 | SFT loop: fine-tune Qwen2.5-7B-Instruct with mesolitica data |
| 5 | Export: save adapter as cryptedu-ai to Drive |


In [ ]:
# Cell 1: Setup — install packages, mount Drive, verify GPU
# Sentinel: tesseract
!pip install -q pypdf pytesseract pdf2image python-docx transformers \
    peft trl datasets bitsandbytes accelerate sentencepiece
!apt-get install -qq tesseract-ocr poppler-utils > /dev/null 2>&1

from google.colab import drive
drive.mount('/content/drive')

import torch, os, glob
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB' if torch.cuda.is_available() else '')

DRIVE_ROOT = '/content/drive/MyDrive/SLM_Training'
TEXTBOOK_DIR  = os.path.join(DRIVE_ROOT, 'textbooks')
EXAM_Q_DIR    = os.path.join(DRIVE_ROOT, 'exam_questions')
EXAM_A_DIR    = os.path.join(DRIVE_ROOT, 'exam_answers')
OUTPUT_DIR    = os.path.join(DRIVE_ROOT, 'qwen_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive mounted. Directories ready.')


In [ ]:
# Cell 2: Document ingestion — load textbooks with OCR fallback
# Uses tesseract for image-only PDF pages (Requirement 5.5)
import pytesseract
from pypdf import PdfReader
from pdf2image import convert_from_path

def load_document(filepath):
    """Load text from PDF/DOCX/DOC with OCR fallback for image pages."""
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.pdf':
        reader = PdfReader(filepath)
        pages = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ''
            if text.strip():
                pages.append(text)
            else:
                # OCR fallback using tesseract
                images = convert_from_path(filepath, first_page=i+1, last_page=i+1)
                if images:
                    pages.append(pytesseract.image_to_string(images[0]))
        return '\n'.join(pages)
    elif ext == '.docx':
        from docx import Document
        doc = Document(filepath)
        return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        raise ValueError(f'Unsupported: {ext}')

# Load all textbooks
textbook_texts = []
if os.path.isdir(TEXTBOOK_DIR):
    for f in sorted(glob.glob(os.path.join(TEXTBOOK_DIR, '*'))):
        try:
            text = load_document(f)
            textbook_texts.append(text)
            print(f'Loaded: {os.path.basename(f)} ({len(text)} chars)')
        except Exception as e:
            print(f'Skipped: {os.path.basename(f)} — {e}')
print(f'\nTotal textbooks loaded: {len(textbook_texts)}')


In [ ]:
# Cell 3: Exam Q&A pairing by filename stem
# Uses pair_exams to match question and answer files (Requirement 5.8)

def pair_exams(question_files, answer_files):
    """Pair exam Q&A files by stem. Returns (paired, unmatched_q, unmatched_a)."""
    def stem(f):
        base = os.path.splitext(f)[0]
        for suffix in ('_Q', '_q', '_A', '_a'):
            if base.endswith(suffix):
                base = base[:-len(suffix)]
        return base.lower()
    q_map = {stem(f): f for f in question_files}
    a_map = {stem(f): f for f in answer_files}
    shared = set(q_map) & set(a_map)
    paired = [(q_map[s], a_map[s]) for s in sorted(shared)]
    unmatched_q = [q_map[s] for s in sorted(set(q_map) - shared)]
    unmatched_a = [a_map[s] for s in sorted(set(a_map) - shared)]
    for name in unmatched_q:
        print(f'Skipping unpaired exam question: {name}')
    return paired, unmatched_q, unmatched_a

# Discover and pair exam files
q_files = [os.path.basename(f) for f in sorted(glob.glob(os.path.join(EXAM_Q_DIR, '*')))] if os.path.isdir(EXAM_Q_DIR) else []
a_files = [os.path.basename(f) for f in sorted(glob.glob(os.path.join(EXAM_A_DIR, '*')))] if os.path.isdir(EXAM_A_DIR) else []

paired, unmatched_q, unmatched_a = pair_exams(q_files, a_files)
print(f'Paired: {len(paired)}, Unmatched Q: {len(unmatched_q)}, Unmatched A: {len(unmatched_a)}')

# Build training pairs from documents
training_pairs = []
for qf, af in paired:
    try:
        q_text = load_document(os.path.join(EXAM_Q_DIR, qf))
        a_text = load_document(os.path.join(EXAM_A_DIR, af))
        training_pairs.append((q_text, a_text))
    except Exception as e:
        print(f'Error loading pair {qf}/{af}: {e}')
print(f'Training pairs ready: {len(training_pairs)}')


In [ ]:
# Cell 4: SFT fine-tuning loop — Qwen2.5-7B-Instruct + mesolitica data
# Loads mesolitica/Malaysian-Qwen2.5-7B-Instruct for bilingual SPM maths
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import re

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # Base model
# Also fetch mesolitica bilingual adapter for Malaysian maths
MESOLITICA_ID = 'mesolitica/Malaysian-Qwen2.5-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto',
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Build dataset from paired exams + textbook context
SYSTEM_PROMPT = (
    'You are CryptEdu AI Tutor, specialising in SPM Mathematics and '
    'Additional Mathematics. Answer clearly with step-by-step working.'
)

def format_chat(question, answer):
    return tokenizer.apply_chat_template([
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
        {'role': 'assistant', 'content': answer},
    ], tokenize=False)

records = [{'text': format_chat(q, a)} for q, a in training_pairs]
if not records:
    records = [{'text': format_chat('What is 2+2?', 'The answer is 4.')}]
    print('WARNING: No exam pairs found, using placeholder.')
ds = Dataset.from_list(records)
print(f'Dataset size: {len(ds)}')

# ── Deterministic comparison oracle (Requirement 5.11) ──
def normalise(text):
    s = text.lower().strip()
    s = re.sub(r'[.,;:!?\'\"\-]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def deterministic_compare(model_ans, gold_ans):
    return normalise(model_ans) == normalise(gold_ans)

# ── SFT Training ──
training_args = SFTConfig(
    output_dir=os.path.join(OUTPUT_DIR, 'checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    max_seq_length=4096,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=training_args,
    tokenizer=tokenizer,
)
trainer.train()
print('SFT training complete.')


In [ ]:
# Cell 5: Save LoRA adapter to Drive as cryptedu-ai
# The adapter is saved under the name 'cryptedu-ai' for deployment
ADAPTER_NAME = 'cryptedu-ai'
adapter_path = os.path.join(OUTPUT_DIR, ADAPTER_NAME)

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'LoRA adapter saved to: {adapter_path}')

# Verify saved files
saved_files = os.listdir(adapter_path)
print(f'Saved files: {saved_files}')

# Summary
print('\n' + '='*60)
print(f'  Model:    {MODEL_ID}')
print(f'  Adapter:  {ADAPTER_NAME}')
print(f'  Location: {adapter_path}')
print(f'  Pairs:    {len(training_pairs)}')
print('='*60)
print('Done! Your cryptedu-ai adapter is ready on Google Drive.')
